# Demo POC local del modelo

**Autora:** Leydy Osorio Vargas  
**Fecha:** 2026-09-09

## Descripción

Este notebook documenta la Tarea 8: una demostración local que carga el pipeline Random Forest
seleccionado, recibe las 13 variables esperadas y genera una predicción reproducible.

La interfaz se implementa en Streamlit mediante `app.py`. Esta POC no reemplaza el despliegue
online y batch solicitado en la evaluación 3.

> **Uso académico:** el resultado no constituye un diagnóstico ni una recomendación médica.

## Alcance y componentes

- `app.py`: formulario y presentación del resultado.
- `src/heart_project/prediction.py`: validación, carga del artefacto e inferencia reutilizable.
- `models/heart_disease_best_pipeline.joblib`: pipeline seleccionado en la Tarea 6.
- `tests/inference/`: pruebas unitarias y pruebas funcionales de la interfaz.

La aplicación limita las entradas a los dominios observados en el dataset y conserva los nombres
internos utilizados durante el entrenamiento.

## 📚 Importar librerías y ubicar el proyecto

In [1]:
import sys
from pathlib import Path

import pandas as pd
import sklearn
import streamlit
from IPython.display import display
from streamlit.testing.v1 import AppTest

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "pyproject.toml").exists():
    PROJECT_DIR = next(
        parent for parent in PROJECT_DIR.parents if (parent / "pyproject.toml").exists()
    )

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from heart_project.prediction import (  # noqa: E402
    FEATURE_COLUMNS,
    load_pipeline,
    predict_patient,
)

MODEL_FILE = PROJECT_DIR / "models" / "heart_disease_best_pipeline.joblib"
APP_FILE = PROJECT_DIR / "app.py"

print(f"Proyecto: {PROJECT_DIR}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"Streamlit: {streamlit.__version__}")

Proyecto: /workspace/scratch/379278d6091f/task8_worktree
Scikit-learn: 1.9.0
Streamlit: 1.63.0


## 🤖 Cargar y verificar el pipeline entrenado

In [2]:
pipeline = load_pipeline(MODEL_FILE)

model_summary = pd.Series(
    {
        "pipeline_steps": " → ".join(pipeline.named_steps),
        "model_class": type(pipeline.named_steps["model"]).__name__,
        "input_features": len(pipeline.feature_names_in_),
        "classes": pipeline.classes_.tolist(),
    },
    name="value",
)
model_summary.to_frame()

,value
pipeline_steps,features → model
model_class,RandomForestClassifier
input_features,13
classes,"[0, 1]"


In [3]:
schema_check = pd.DataFrame(
    {
        "position": range(1, len(FEATURE_COLUMNS) + 1),
        "application_field": FEATURE_COLUMNS,
        "model_field": pipeline.feature_names_in_,
    }
)
schema_check["matches"] = schema_check["application_field"] == schema_check["model_field"]
schema_check

,position,application_field,model_field,matches
0,1,age,age,True
1,2,sex,sex,True
2,3,chest_pain,chest_pain,True
3,4,rest_bp,rest_bp,True
4,5,chol,chol,True
5,6,fbs,fbs,True
6,7,rest_ecg,rest_ecg,True
7,8,max_hr,max_hr,True
8,9,exang,exang,True
9,10,old_peak,old_peak,True


## 🧪 Ejecutar una predicción de ejemplo

El registro siguiente utiliza valores válidos dentro de los dominios documentados. Su propósito
es demostrar el flujo técnico; no representa una persona real.

In [4]:
example_patient = {
    "age": 54,
    "sex": "Male",
    "chest_pain": "asymptomatic",
    "rest_bp": 130,
    "chol": 241,
    "fbs": False,
    "rest_ecg": "normal",
    "max_hr": 153,
    "exang": False,
    "old_peak": 0.8,
    "slope": 2,
    "ca": 0,
    "thal": "normal",
}

pd.Series(example_patient, name="value").to_frame()

,value
age,54
sex,Male
chest_pain,asymptomatic
rest_bp,130
chol,241
fbs,False
rest_ecg,normal
max_hr,153
exang,False
old_peak,0.8


In [5]:
prediction = predict_patient(pipeline, example_patient)

prediction_summary = pd.Series(
    {
        "predicted_class": prediction.predicted_class,
        "class_interpretation": (
            "patrón compatible con clase positiva"
            if prediction.has_disease_pattern
            else "patrón compatible con clase negativa"
        ),
        "positive_class_probability": prediction.disease_probability,
        "decision_threshold": prediction.threshold,
    },
    name="result",
)
prediction_summary.to_frame()

,result
predicted_class,0
class_interpretation,patrón compatible con clase negativa
positive_class_probability,0.194486
decision_threshold,0.5


## 🖥️ Evidencia funcional de la interfaz

`AppTest` ejecuta la aplicación completa, envía el formulario con sus valores predeterminados y
comprueba que Streamlit presente una predicción sin excepciones. La captura visual puede obtenerse
al abrir el puerto local `8501` en Codespaces.

In [6]:
app_test = AppTest.from_file(str(APP_FILE)).run(timeout=30)
app_test.button[0].click().run(timeout=30)

ui_message = app_test.success[0].value if app_test.success else app_test.warning[0].value
ui_evidence = pd.Series(
    {
        "title": app_test.title[0].value,
        "number_inputs": len(app_test.number_input),
        "select_inputs": len(app_test.selectbox),
        "checkbox_inputs": len(app_test.checkbox),
        "result_message": ui_message,
        "probability_metric": app_test.metric[0].value,
        "threshold_metric": app_test.metric[1].value,
        "exceptions": len(app_test.exception),
    },
    name="observed_value",
)
ui_evidence.to_frame()

,observed_value
title,❤️ Demo POC del modelo de enfermedad cardiaca
number_inputs,5
select_inputs,6
checkbox_inputs,2
result_message,El modelo identifica un patrón compatible con ...
probability_metric,17.3%
threshold_metric,50%
exceptions,0


## ▶️ Instrucciones de ejecución local

Desde la raíz del proyecto:

```bash
uv sync
uv run streamlit run app.py
```

En GitHub Codespaces:

1. Esperar el mensaje que indica que Streamlit está disponible.
2. Abrir la pestaña **Ports** o **Puertos**.
3. Buscar el puerto `8501` y seleccionar **Open in Browser**.
4. Completar el formulario y pulsar **Generar predicción**.

Para ejecutar las pruebas:

```bash
uv run pytest
uv run pre-commit run --all-files
```

## ✅ Validación final de entregables

In [7]:
expected_feature_count = 13
expected_metric_count = 2

validation_checks = pd.Series(
    {
        "model_artifact_exists": MODEL_FILE.is_file(),
        "streamlit_app_exists": APP_FILE.is_file(),
        "application_has_13_fields": len(FEATURE_COLUMNS) == expected_feature_count,
        "application_schema_matches_model": schema_check["matches"].all(),
        "probability_is_valid": 0.0 <= prediction.disease_probability <= 1.0,
        "streamlit_started_without_exceptions": not app_test.exception,
        "streamlit_generated_prediction": len(app_test.metric) == expected_metric_count,
    },
    name="passed",
)

display(validation_checks.to_frame())
assert validation_checks.all()
print("Validación final: todos los controles fueron aprobados.")

                                      passed
model_artifact_exists                   True
streamlit_app_exists                    True
application_has_13_fields               True
application_schema_matches_model        True
probability_is_valid                    True
streamlit_started_without_exceptions    True
streamlit_generated_prediction          True
Validación final: todos los controles fueron aprobados.


## Conclusiones

- El pipeline persistido puede cargarse sin reentrenar el modelo.
- La aplicación recibe exactamente las 13 variables requeridas y valida sus dominios.
- La predicción incluye clase, probabilidad positiva y umbral utilizado.
- Las pruebas funcionales confirman que la interfaz inicia y procesa el formulario sin errores.
- El resultado se comunica con lenguaje prudente y un aviso explícito de uso no clínico.
- La separación entre interfaz e inferencia facilita reutilizar la lógica en el futuro despliegue
  online y batch.